# Superstore Sales Analysis using SQL
### Filtering, Aggregation,Business & Validation Queries

Analyzing the Superstore retail dataset with SQL.
The CSV is loaded into a SQLite database via pandas, and each step runs SQL
queries to pull out business insights.

**Steps:** load → explore → filter (WHERE) → aggregate (GROUP BY) →
sort & limit → business use cases → validation → insights.

# Step 1: Importing Required Libraries


In [27]:
import pandas as pd
import sqlite3
print("Libraries Imported")

Libraries Imported


# Step 2 Load the Dataset(Superstore.csv)

In [28]:
df = pd.read_csv('Dataset/Sample - Superstore.csv',encoding="latin-1")

# Parse the date columns and store as ISO text (YYYY-MM-DD) for reliable SQL date queries
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format="%m/%d/%Y").dt.strftime("%Y-%m-%d")

df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200


# DataSet Overview 
- Gives idea about structure of dataset


In [29]:
df.shape
print("Rows:",df.shape[0])
print("Columns:",df.shape[1])

Rows: 9994
Columns: 21


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [31]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

# Step 4 Connect to SQLite and load the table

In [32]:
conn = sqlite3.connect("superstore.db")


In [33]:
#Load the Dataframe into sql table
df.to_sql("superstore",conn, if_exists="replace", index = False)


9994

In [34]:
#Helper to run the sql queries and return the dataframe as output
def run_query(query):
    return pd.read_sql(query,conn)
print("Dataset loaded into SQL successfully")

Dataset loaded into SQL successfully


# Step 5 Exploring the table(Schema & sample Data)


In [35]:
#Schema
run_query("PRAGMA table_info(superstore)")[["name", "type"]]


,name,type
0,Row ID,INTEGER
1,Order ID,TEXT
2,Order Date,TEXT
3,Ship Date,TEXT
4,Ship Mode,TEXT
5,Customer ID,TEXT
6,Customer Name,TEXT
7,Segment,TEXT
8,Country,TEXT
9,City,TEXT


In [36]:
#Sample data first 5 rows
run_query("SELECT * FROM superstore LIMIT 5")


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [37]:
# Quick size summary
run_query("""
SELECT
    COUNT(*)  AS total_rows,
    COUNT(DISTINCT "Order ID")  AS distinct_orders,
    COUNT(DISTINCT "Customer ID") AS distinct_customers,
    MIN("Order Date") AS first_order,
    MAX("Order Date") AS last_order
FROM superstore
""")

,total_rows,distinct_orders,distinct_customers,first_order,last_order
0,9994,5009,793,2014-01-03,2017-12-30


# Step 6 Filtering with WHERE clause


In [38]:
# Filtering by region
run_query("""
SELECT "Order ID", "Customer Name", Category, Sales, Profit, Region
FROM superstore
WHERE Region = 'West'
LIMIT 10
""")

,Order ID,Customer Name,Category,Sales,Profit,Region
0,CA-2016-138688,Darrin Van Huff,Office Supplies,14.620,6.8714,West
1,CA-2014-115812,Brosina Hoffman,Furniture,48.860,14.1694,West
2,CA-2014-115812,Brosina Hoffman,Office Supplies,7.280,1.9656,West
3,CA-2014-115812,Brosina Hoffman,Technology,907.152,90.7152,West
4,CA-2014-115812,Brosina Hoffman,Office Supplies,18.504,5.7825,West
5,CA-2014-115812,Brosina Hoffman,Office Supplies,114.900,34.4700,West
6,CA-2014-115812,Brosina Hoffman,Furniture,1706.184,85.3092,West
7,CA-2014-115812,Brosina Hoffman,Technology,911.424,68.3568,West
8,CA-2016-161389,Irene Maddox,Office Supplies,407.976,132.5922,West
9,CA-2014-167164,Alejandro Grove,Office Supplies,55.500,9.9900,West


In [39]:
# Filtering by category
run_query("""
SELECT "Order ID", "Product Name", Sales, Profit,Category
FROM superstore
WHERE Category = 'Technology'
LIMIT 10
""")

,Order ID,Product Name,Sales,Profit,Category
0,CA-2014-115812,Mitel 5320 IP Phone VoIP phone,907.152,90.7152,Technology
1,CA-2014-115812,Konftel 250 Conference phone - Charcoal black,911.424,68.3568,Technology
2,CA-2014-143336,Cisco SPA 501G IP Phone,213.480,16.0110,Technology
3,CA-2016-121755,Imation 8GB Mini TravelDrive USB 2.0 Flash Drive,90.570,11.7741,Technology
4,CA-2016-117590,GE 30524EE4,1097.544,123.4737,Technology
5,CA-2015-117415,Plantronics HL10 Handset Lifter,371.168,41.7564,Technology
6,CA-2017-120999,Panasonic Kx-TS550,147.168,16.5564,Technology
7,CA-2016-118255,Verbatim 25 GB 6x Blu-ray Single Layer Recorda...,45.980,19.7714,Technology
8,CA-2016-169194,Imation 8gb Micro Traveldrive Usb 2.0 Flash Drive,45.000,4.9500,Technology
9,CA-2016-169194,"LF Elite 3D Dazzle Designer Hard Case Cover, L...",21.800,6.1040,Technology


In [40]:
#Filtering by sales value 
run_query("""
SELECT "Order ID", "Product Name", Sales
FROM superstore
WHERE Sales > 2000
ORDER BY Sales DESC
LIMIT 10
""")

,Order ID,Product Name,Sales
0,CA-2014-145317,Cisco TelePresence System EX90 Videoconferenci...,22638.480
1,CA-2016-118689,Canon imageCLASS 2200 Advanced Copier,17499.950
2,CA-2017-140151,Canon imageCLASS 2200 Advanced Copier,13999.960
3,CA-2017-127180,Canon imageCLASS 2200 Advanced Copier,11199.968
4,CA-2017-166709,Canon imageCLASS 2200 Advanced Copier,10499.970
5,CA-2016-117121,GBC Ibimaster 500 Manual ProClick Binding System,9892.740
6,CA-2014-116904,Ibico EPK-21 Electric Binding System,9449.950
7,US-2016-107440,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930
8,CA-2016-158841,HP Designjet T520 Inkjet Large Format Printer ...,8749.950
9,CA-2016-143714,Canon imageCLASS 2200 Advanced Copier,8399.976


In [41]:
#Filering by Date - all the orders that are placed in 2017
run_query("""
SELECT "Order ID","Order Date", "Customer Name", "Product ID","Category", "Prdouct Name"
FROM superstore
WHERE "Order Date" >= '2017-01-01' AND "Order Date" < '2018-01-01'
""")

,Order ID,Order Date,Customer Name,Product ID,Category,"""Prdouct Name"""
0,CA-2017-114412,2017-04-15,Andrew Allen,OFF-PA-10002365,Office Supplies,Prdouct Name
1,US-2017-156909,2017-07-16,Sandra Flanagan,FUR-CH-10002774,Furniture,Prdouct Name
2,CA-2017-107727,2017-10-19,Matt Abelman,OFF-PA-10000249,Office Supplies,Prdouct Name
3,CA-2017-120999,2017-09-10,Linda Cazamias,TEC-PH-10004093,Technology,Prdouct Name
4,CA-2017-139619,2017-09-19,Erin Smith,OFF-ST-10003282,Office Supplies,Prdouct Name
...,...,...,...,...,...,...
3307,CA-2017-163629,2017-11-17,Ruben Ausman,TEC-PH-10004006,Technology,Prdouct Name
3308,CA-2017-121258,2017-02-26,Dave Brooks,FUR-FU-10000747,Furniture,Prdouct Name
3309,CA-2017-121258,2017-02-26,Dave Brooks,TEC-PH-10003645,Technology,Prdouct Name
3310,CA-2017-121258,2017-02-26,Dave Brooks,OFF-PA-10004041,Office Supplies,Prdouct Name


**Insight:** Filtering lets us isolate a region, product category, value band, or
time window so each business question works on exactly the relevant slice of data.

# Step 7 Aggregation with GROUP BY  
Aggregation collapses thousands of rows into business metrics — totals, counts,
and averages per group.

In [42]:
# total sales by region

run_query("""
SELECT Region, ROUND(SUM(Sales), 2) AS Total_Sales
FROM superstore
GROUP BY Region
ORDER BY Total_Sales DESC
""")

,Region,Total_Sales
0,West,725457.82
1,East,678781.24
2,Central,501239.89
3,South,391721.91


In [43]:
# Total quantity sold by category

run_query("""
SELECT Category, SUM(Quantity) AS Total_Quantity
FROM superstore
GROUP BY Category
ORDER BY Total_Quantity DESC
""")

,Category,Total_Quantity
0,Office Supplies,22906
1,Furniture,8028
2,Technology,6939


In [44]:
#Average sales and average profit by category
run_query("""
SELECT Category,
       ROUND(AVG(Sales), 2)  AS Avg_Sales,
       ROUND(AVG(Profit), 2) AS Avg_Profit
FROM superstore
GROUP BY Category
""")

,Category,Avg_Sales,Avg_Profit
0,Furniture,349.83,8.70
1,Office Supplies,119.32,20.33
2,Technology,452.71,78.75


## Step 8: Sort & limit (top products, top categories)


In [45]:
# Top 10 products by total sales
run_query("""
SELECT "Product Name",
       ROUND(SUM(Sales), 2)  AS Total_Sales,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM superstore
GROUP BY "Product Name"
ORDER BY Total_Sales DESC
LIMIT 10
""") 

,Product Name,Total_Sales,Total_Profit
0,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,0.00
4,GBC DocuBind TL300 Electric Binding System,19823.48,2233.51
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98
6,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98
8,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17
9,High Speed Automatic Electric Letter Opener,17030.31,-262.00


In [46]:
# Categories ranked by total profit
run_query("""
SELECT Category, ROUND(SUM(Profit), 2) AS Total_Profit
FROM superstore
GROUP BY Category
ORDER BY Total_Profit DESC
""")

,Category,Total_Profit
0,Technology,145454.95
1,Office Supplies,122490.80
2,Furniture,18451.27


## Step 9: Business use cases


In [47]:
# Monthly sales trend (year-month)
run_query("""
SELECT substr("Order Date", 1, 7) AS Year_Month,
       ROUND(SUM(Sales), 2)       AS Monthly_Sales,
       ROUND(SUM(Profit), 2)      AS Monthly_Profit
FROM superstore
GROUP BY Year_Month
ORDER BY Year_Month
LIMIT 12
""")

,Year_Month,Monthly_Sales,Monthly_Profit
0,2014-01,14236.90,2450.19
1,2014-02,4519.89,862.31
2,2014-03,55691.01,498.73
3,2014-04,28295.35,3488.84
4,2014-05,23648.29,2738.71
5,2014-06,34595.13,4976.52
6,2014-07,33946.39,-841.48
7,2014-08,27909.47,5318.10
8,2014-09,81777.35,8328.10
9,2014-10,31453.39,3448.26


In [48]:
# Top 10 customers by total spend
run_query("""
SELECT "Customer Name",
       COUNT(DISTINCT "Order ID") AS Orders,
       ROUND(SUM(Sales), 2)       AS Total_Spend
FROM superstore
GROUP BY "Customer ID", "Customer Name"
ORDER BY Total_Spend DESC
LIMIT 10
""")

,Customer Name,Orders,Total_Spend
0,Sean Miller,5,25043.05
1,Tamara Chand,5,19052.22
2,Raymond Buch,6,15117.34
3,Tom Ashbrook,4,14595.62
4,Adrian Barton,10,14473.57
5,Ken Lonsdale,12,14175.23
6,Sanjit Chand,9,14142.33
7,Hunter Lopez,6,12873.30
8,Sanjit Engle,11,12209.44
9,Christopher Conant,5,12129.07


In [49]:
# Duplicate check — does any Order and Product line repeat?

run_query("""
SELECT "Order ID", "Product ID", COUNT(*) AS times_appearing
FROM superstore
GROUP BY "Order ID", "Product ID"
HAVING COUNT(*) > 1
ORDER BY times_appearing DESC
""")

,Order ID,Product ID,times_appearing
0,CA-2015-103135,OFF-BI-10000069,2
1,CA-2016-129714,OFF-PA-10001970,2
2,CA-2016-137043,FUR-FU-10003664,2
3,CA-2016-140571,OFF-PA-10001954,2
4,CA-2017-118017,TEC-AC-10002006,2
5,CA-2017-152912,OFF-ST-10003208,2
6,US-2014-150119,FUR-CH-10002965,2
7,US-2016-123750,TEC-AC-10004659,2


**Note on the duplicate check:** a few Order and Product combinations repeat. These are
not bad data as the same product can appear on one order as separate line items
(e.g. billed at different discounts). The Row ID stays unique, so totals are reliable.

## Step 10: Validate results (row counts & data quality)


In [50]:
# Missing values in key columns
run_query("""
SELECT
    SUM(CASE WHEN "Order ID"  IS NULL THEN 1 ELSE 0 END) AS missing_order_id,
    SUM(CASE WHEN Sales       IS NULL THEN 1 ELSE 0 END) AS missing_sales,
    SUM(CASE WHEN "Order Date" IS NULL THEN 1 ELSE 0 END) AS missing_order_date
FROM superstore
""")

,missing_order_id,missing_sales,missing_order_date
0,0,0,0


In [51]:
# Reconcile: do per-category sales sum to the grand total?
by_cat = run_query("SELECT ROUND(SUM(Sales),2) AS s FROM superstore GROUP BY Category")
grand  = run_query("SELECT ROUND(SUM(Sales),2) AS s FROM superstore")
print("Sum of category sales:", round(by_cat['s'].sum(), 2))
print("Grand total sales:", grand['s'].iloc[0])

Sum of category sales: 2297200.86
Grand total sales: 2297200.86


In [52]:
# Distinct customers 
run_query('SELECT COUNT(DISTINCT "Customer ID") AS total_customers FROM superstore')

,total_customers
0,793


## Insights & Summary

- Technology and Furniture lead total sales, but profit differs sharply by category —
  Furniture's profit is thin relative to its sales.
- The West and East regions generate the most profit; Central trails despite solid sales.
- A small group of customers accounts for a large share of total spend (top-10 list above).
- Sales rise toward the end of each year, showing a seasonal Q4 pattern in the monthly trend.
- Data quality is solid: no nulls in key fields, category sales reconcile to the grand
  total, and the only repeated Order+Product rows are legitimate split line items.